In [ ]:
# -*- coding: utf-8 -*-
!pip install catboost

import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold

# ==========================================
# 1. ЗАГРУЗКА И ОЧИСТКА
# ==========================================
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

train_data = train_data.rename(columns={'IC50, mM': 'IC50', 'CC50, mM': 'CC50'})
if 'IC50, mM' in test_data.columns:
    test_data = test_data.rename(columns={'IC50, mM': 'IC50', 'CC50, mM': 'CC50'})

train_data = train_data.fillna(train_data.median(numeric_only=True))
test_data = test_data.fillna(test_data.median(numeric_only=True))

constant_cols = [col for col in train_data.columns if train_data[col].nunique() <= 1]
train_features = train_data.drop(columns=constant_cols + ['index', 'IC50', 'CC50', 'SI'], errors='ignore')
test_features = test_data.drop(columns=constant_cols + ['index'], errors='ignore')

# ==========================================
# 2. ОТБОР ПРИЗНАКОВ (Как в версии на 279)
# ==========================================
targets_all = ['IC50', 'CC50', 'SI']
important_features_list = []

print("Отбираем признаки на сырых данных...")
for target in targets_all:
    y_raw = train_data[target]

    cb_fs = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6, loss_function='MAE', random_seed=42, verbose=0)
    cb_fs.fit(train_features, y_raw)

    fi = pd.DataFrame({'Feature': train_features.columns, 'Importance': cb_fs.get_feature_importance()})
    top_30 = fi.sort_values(by='Importance', ascending=False).head(30)['Feature'].tolist()
    important_features_list.extend(top_30)

best_features = list(set(important_features_list))
print(f"Отобрано уникальных признаков: {len(best_features)}")

X_selected = train_features[best_features]
test_selected = test_features[best_features]

# ==========================================
# 3. ОБУЧЕНИЕ: 5 СИДОВ + MAE (Без вычислений SI по формуле!)
# ==========================================
is_good_drug = (train_data['SI'] > 8).astype(int)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

test_preds = {t: np.zeros(len(test_selected)) for t in targets_all}
n_seeds = 5

print(f"\nОбучаем модели с функцией потерь MAE (Усреднение {n_seeds} раз)...")
for seed in range(n_seeds):
    print(f"--- Итерация {seed + 1}/{n_seeds} ---")
    for target in targets_all:
        y_raw = train_data[target]

        for fold, (train_idx, val_idx) in enumerate(skf.split(X_selected, is_good_drug)):
            X_train, y_train = X_selected.iloc[train_idx], y_raw.iloc[train_idx]
            X_val, y_val = X_selected.iloc[val_idx], y_raw.iloc[val_idx]

            model = CatBoostRegressor(
                iterations=1500,
                learning_rate=0.03,
                depth=6,
                l2_leaf_reg=7,
                loss_function='MAE',     # Сохраняем MAE!
                eval_metric='RMSE',
                random_seed=42 + seed * 100 + fold, # Разнообразие деревьев
                verbose=0
            )

            model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=150)

            # Накапливаем сырые предсказания
            test_preds[target] += model.predict(test_selected) / (skf.n_splits * n_seeds)

# ==========================================
# 4. ЖЕСТКИЙ КЛИППИНГ (Возвращаем 98 перцентиль)
# ==========================================
# Этот блок спас нас в версии 279. Он жестко отсекает любые космические выбросы.
clip_limits = {
    'IC50': train_data['IC50'].quantile(0.98),
    'CC50': train_data['CC50'].quantile(0.98),
    'SI': train_data['SI'].quantile(0.98)
}

submission_raw = pd.DataFrame({'index': test_data['index']})

for target in targets_all:
    # 1. Защита от отрицательных чисел и нулей
    preds = np.maximum(0.001, test_preds[target])
    # 2. Защита от гигантских выбросов по верхней границе трейна
    preds = np.minimum(preds, clip_limits[target])

    submission_raw[target] = preds

submission_raw.to_csv('submission_best_mae_seeds.csv', index=False)
print("\nФайл 'submission_best_mae_seeds.csv' готов. Отправляем!")